In [153]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)

from xgboost import XGBRegressor, XGBClassifier

In [154]:
sales=pd.read_csv('sales_daily.csv')
inventory=pd.read_csv('inventory_snapshots.csv')
calender=pd.read_csv('calendar.csv')
sku_master=pd.read_csv('sku_master.csv')

In [155]:
# convert date columns
sales['Date']=pd.to_datetime(sales['Date'])

inventory['Snapshot_Date']=pd.to_datetime(inventory['Snapshot_Date'])

calender['date']=pd.to_datetime(calender['date'])

sku_master['Launch_Date']=pd.to_datetime(sku_master['Launch_Date'])

In [156]:
# Inspect the datasets
display(sales.head())
display(inventory.head())
display(calendar.head())
display(sku_master.head())

,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0


,Snapshot_Date,SKU,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value
0,2024-01-01,SKU001,16,23,11,7,18,58420.96
1,2024-01-01,SKU002,29,12,4,5,10,163983.40
2,2024-01-01,SKU003,34,23,14,6,20,209060.22
3,2024-01-01,SKU004,34,4,7,5,12,42257.92
4,2024-01-01,SKU005,23,5,5,6,11,170945.89


,date,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event
0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
1,2024-01-02,2024,1,Q1,1,Tuesday,0,Winter,NaN,0,NaN
2,2024-01-03,2024,1,Q1,1,Wednesday,0,Winter,NaN,0,NaN
3,2024-01-04,2024,1,Q1,1,Thursday,0,Winter,NaN,0,NaN
4,2024-01-05,2024,1,Q1,1,Friday,0,Winter,NaN,0,NaN


,SKU,Product_Name,Category,Subcategory,Launch_Date,Cost_Price,Selling_Price,Gross_Margin_Per_Unit
0,SKU001,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.60
1,SKU002,Product 002,Home Decor,Table,2024-05-01,3867.09,3805.69,-61.40
2,SKU003,Product 003,Kitchen,Cushion,2023-12-22,589.48,8078.23,7488.75
3,SKU004,Product 004,Lighting,Cookware,2023-04-28,1445.74,6861.57,5415.83
4,SKU005,Product 005,Storage,Lamp,2023-04-22,5543.63,9493.22,3949.59


In [157]:
#Stardardize SKU
sales['SKU']=sales['SKU'].astype(str).str.strip()
inventory['SKU']=inventory['SKU'].astype(str).str.strip()
sku_master['SKU']=sku_master['SKU'].astype(str).str.strip()

In [158]:
# Merge sales + SKU master + calendar
df = sales.merge(
    sku_master,
    on="SKU",
    how="left"
)

df = df.merge(
    calendar,
    left_on="Date",
    right_on="date",
    how="left"
)

df.drop(columns=["date"], inplace=True)

In [159]:
#merge monthly inventory snapshot
df["inventory_month"] = df["Date"].dt.to_period("M")

inventory_monthly = inventory.copy()
inventory_monthly["inventory_month"] = (
    inventory_monthly["Snapshot_Date"].dt.to_period("M")
)

inventory_monthly = inventory_monthly.drop(
    columns=["Snapshot_Date"]
)

df = df.merge(
    inventory_monthly,
    on=["SKU", "inventory_month"],
    how="left",
    validate="many_to_one"
)

df.drop(columns=["inventory_month"], inplace=True)

df = df.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

print("Final combined raw dataset:", df.shape)

Final combined raw dataset: (36550, 29)


In [160]:
#create date feature
df["day"] = df["Date"].dt.day
df["day_of_year"] = df["Date"].dt.dayofyear
df["month_num"] = df["Date"].dt.month
df["year_num"] = df["Date"].dt.year

df["product_age_days"] = (
    df["Date"] - df["Launch_Date"]
).dt.days.clip(lower=0)

# Historical demand features
g = df.groupby("SKU")["Units_Sold"]

df["lag_1"] = g.shift(1)
df["lag_7"] = g.shift(7)
df["lag_14"] = g.shift(14)
df["lag_28"] = g.shift(28)

df["rolling_7"] = (
    g.shift(1)
     .rolling(7)
     .mean()
     .reset_index(level=0, drop=True)
)

df["rolling_14"] = (
    g.shift(1)
     .rolling(14)
     .mean()
     .reset_index(level=0, drop=True)
)

df["rolling_28"] = (
    g.shift(1)
     .rolling(28)
     .mean()
     .reset_index(level=0, drop=True)
)

df["rolling_7_std"] = (
    g.shift(1)
     .rolling(7)
     .std()
     .reset_index(level=0, drop=True)
)

df["rolling_7_std"] = df["rolling_7_std"].fillna(0)

In [161]:
#create future 7-day demand target
future_cols = []

for i in range(1, 8):
    col = f"future_{i}"
    df[col] = g.shift(-i)
    future_cols.append(col)

df["future_7d_demand"] = (
    df[future_cols]
    .sum(axis=1, min_count=7)
)

display(
    df[
        ["SKU", "Date", "Units_Sold"]
        + future_cols
        + ["future_7d_demand"]
    ].head(10)
)

,SKU,Date,Units_Sold,future_1,future_2,future_3,future_4,future_5,future_6,future_7,future_7d_demand
0,SKU001,2024-01-01,5,13.0,12.0,23.0,16.0,18.0,19.0,12.0,113.0
1,SKU001,2024-01-02,13,12.0,23.0,16.0,18.0,19.0,12.0,10.0,110.0
2,SKU001,2024-01-03,12,23.0,16.0,18.0,19.0,12.0,10.0,11.0,109.0
3,SKU001,2024-01-04,23,16.0,18.0,19.0,12.0,10.0,11.0,11.0,97.0
4,SKU001,2024-01-05,16,18.0,19.0,12.0,10.0,11.0,11.0,14.0,95.0
5,SKU001,2024-01-06,18,19.0,12.0,10.0,11.0,11.0,14.0,10.0,87.0
6,SKU001,2024-01-07,19,12.0,10.0,11.0,11.0,14.0,10.0,17.0,85.0
7,SKU001,2024-01-08,12,10.0,11.0,11.0,14.0,10.0,17.0,15.0,88.0
8,SKU001,2024-01-09,10,11.0,11.0,14.0,10.0,17.0,15.0,18.0,96.0
9,SKU001,2024-01-10,11,11.0,14.0,10.0,17.0,15.0,18.0,21.0,106.0


In [162]:
#target 2 stockout risk
df["available_inventory"] = (
    df["Current_Stock"].fillna(0)
    + df["On_Order"].fillna(0)
)

df["stockout_risk_target"] = (
    df["available_inventory"]
    < df["future_7d_demand"]
).astype(int)

In [163]:
#recomended reorder priority
df["reorder_qty_target"] = (
    df["future_7d_demand"]
    + df["Safety_Stock"].fillna(0)
    - df["Current_Stock"].fillna(0)
    - df["On_Order"].fillna(0)
).clip(lower=0)

display(
    df[
        [
            "SKU",
            "Date",
            "Current_Stock",
            "On_Order",
            "Safety_Stock",
            "future_7d_demand",
            "stockout_risk_target",
            "reorder_qty_target"
        ]
    ].head(10)
)

,SKU,Date,Current_Stock,On_Order,Safety_Stock,future_7d_demand,stockout_risk_target,reorder_qty_target
0,SKU001,2024-01-01,16,23,7,113.0,1,81.0
1,SKU001,2024-01-02,16,23,7,110.0,1,78.0
2,SKU001,2024-01-03,16,23,7,109.0,1,77.0
3,SKU001,2024-01-04,16,23,7,97.0,1,65.0
4,SKU001,2024-01-05,16,23,7,95.0,1,63.0
5,SKU001,2024-01-06,16,23,7,87.0,1,55.0
6,SKU001,2024-01-07,16,23,7,85.0,1,53.0
7,SKU001,2024-01-08,16,23,7,88.0,1,56.0
8,SKU001,2024-01-09,16,23,7,96.0,1,64.0
9,SKU001,2024-01-10,16,23,7,106.0,1,74.0


In [164]:
#remove rows where historical feature unavailable
required_columns = [
    "future_7d_demand",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_7",
    "rolling_14",
    "rolling_28",
    "Current_Stock",
    "On_Order",
    "Lead_Time_Days",
    "Safety_Stock",
]

model_df = df.dropna(
    subset=required_columns
).copy()

model_df["rolling_7_std"] = (
    model_df["rolling_7_std"].fillna(0)
)

print("Modeling dataset:", model_df.shape)
print("SKUs:", model_df["SKU"].nunique())
display(model_df.head())

Modeling dataset: (34800, 53)
SKUs: 50


,Date,SKU,Units_Sold,Revenue,Price,Promotion,Product_Name,Category,Subcategory,Launch_Date,...,future_2,future_3,future_4,future_5,future_6,future_7,future_7d_demand,available_inventory,stockout_risk_target,reorder_qty_target
28,2024-01-29,SKU001,14,51296.70,3664.05,0,Product 001,Furniture,Chair,2022-04-09,...,7.0,12.0,21.0,24.0,18.0,15.0,120.0,39,1,88.0
29,2024-01-30,SKU001,23,84273.15,3664.05,0,Product 001,Furniture,Chair,2022-04-09,...,12.0,21.0,24.0,18.0,15.0,18.0,115.0,39,1,83.0
30,2024-01-31,SKU001,7,25648.35,3664.05,0,Product 001,Furniture,Chair,2022-04-09,...,21.0,24.0,18.0,15.0,18.0,16.0,124.0,39,1,92.0
31,2024-02-01,SKU001,12,43968.60,3664.05,0,Product 001,Furniture,Chair,2022-04-09,...,24.0,18.0,15.0,18.0,16.0,19.0,131.0,1414,0,0.0
32,2024-02-02,SKU001,21,76945.05,3664.05,0,Product 001,Furniture,Chair,2022-04-09,...,18.0,15.0,18.0,16.0,19.0,10.0,120.0,1414,0,0.0


In [165]:
#define model feature
categorical_features = [
    "SKU",
    "Category",
    "Subcategory",
    "quarter",
    "day_of_week",
    "season",
    "holiday",
    "promotion_event",
]

numeric_features = [
    "Price",
    "Promotion",
    "year",
    "month",
    "week",
    "is_weekend",
    "is_holiday",
    "day",
    "day_of_year",
    "month_num",
    "year_num",
    "product_age_days",
    "Cost_Price",
    "Selling_Price",
    "Gross_Margin_Per_Unit",
    "Current_Stock",
    "On_Order",
    "Lead_Time_Days",
    "Safety_Stock",
    "Reorder_Point",
    "Inventory_Value",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_7",
    "rolling_14",
    "rolling_28",
    "rolling_7_std",
]

feature_columns = categorical_features + numeric_features

X = model_df[feature_columns].copy()

for col in categorical_features:
    X[col] = (
        X[col]
        .fillna("Unknown")
        .astype(str)
    )

for col in numeric_features:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    ).fillna(0)

print("Number of features before encoding:", len(feature_columns))

Number of features before encoding: 37


In [166]:
#split 80/20
dates = np.sort(
    model_df["Date"].unique()
)

split_index = int(
    len(dates) * 0.80
)

split_date = pd.Timestamp(
    dates[split_index]
)

train_mask = model_df["Date"] < split_date
test_mask = model_df["Date"] >= split_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_demand_train = model_df.loc[
    train_mask,
    "future_7d_demand"
]

y_demand_test = model_df.loc[
    test_mask,
    "future_7d_demand"
]

y_risk_train = model_df.loc[
    train_mask,
    "stockout_risk_target"
]

y_risk_test = model_df.loc[
    test_mask,
    "stockout_risk_target"
]

y_reorder_train = model_df.loc[
    train_mask,
    "reorder_qty_target"
]

y_reorder_test = model_df.loc[
    test_mask,
    "reorder_qty_target"
]

print("Split date:", split_date.date())
print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

Split date: 2025-08-07
Training rows: 27800
Testing rows : 7000


In [167]:
#preprocessor factory
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                categorical_features,
            ),
            (
                "num",
                "passthrough",
                numeric_features,
            ),
        ]
    )


In [145]:
#model 1 7-day demand forecast
demand_model = Pipeline([
    (
        "preprocessor",
        make_preprocessor()
    ),
    (
        "model",
        XGBRegressor(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1
        )
    )
])

demand_model.fit(
    X_train,
    y_demand_train
)

demand_pred = demand_model.predict(
    X_test
)

demand_mae = mean_absolute_error(
    y_demand_test,
    demand_pred
)

demand_rmse = np.sqrt(
    mean_squared_error(
        y_demand_test,
        demand_pred
    )
)

demand_r2 = r2_score(
    y_demand_test,
    demand_pred
)

print("XGBoost Demand Forecast")
print("MAE :", round(demand_mae, 3))
print("RMSE:", round(demand_rmse, 3))
print("R2  :", round(demand_r2, 3))
print("R2 %:", round(demand_r2 * 100, 2))

XGBoost Demand Forecast
MAE : 8.082
RMSE: 10.452
R2  : 0.944
R2 %: 94.36


In [146]:
#model 2 stockout risk
# Calculate class imbalance weight from the training data
negative = (y_risk_train == 0).sum()
positive = (y_risk_train == 1).sum()

scale_pos_weight = (
    negative / positive
    if positive > 0
    else 1.0
)

print("Scale Pos Weight:", round(scale_pos_weight, 3))

risk_model = Pipeline([
    (
        "preprocessor",
        make_preprocessor()
    ),
    (
        "model",
        XGBClassifier(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1
        )
    )
])

risk_model.fit(
    X_train,
    y_risk_train
)

risk_pred = risk_model.predict(
    X_test
)

risk_probability = (
    risk_model.predict_proba(X_test)[:, 1]
)

risk_accuracy = accuracy_score(
    y_risk_test,
    risk_pred
)

print("XGBoost Stockout Risk Accuracy:",
      round(risk_accuracy, 3))

print("Accuracy %:",
      round(risk_accuracy * 100, 2))

print("\nClassification Report:")
print(
    classification_report(
        y_risk_test,
        risk_pred
    )
)

Scale Pos Weight: 16.506
XGBoost Stockout Risk Accuracy: 0.99
Accuracy %: 99.03

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      6661
           1       0.92      0.87      0.90       339

    accuracy                           0.99      7000
   macro avg       0.96      0.93      0.95      7000
weighted avg       0.99      0.99      0.99      7000



In [147]:
#model 3 reorder quantity
reorder_model = Pipeline([
    (
        "preprocessor",
        make_preprocessor()
    ),
    (
        "model",
        XGBRegressor(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1
        )
    )
])

reorder_model.fit(
    X_train,
    y_reorder_train
)

reorder_pred = reorder_model.predict(
    X_test
)

reorder_pred = np.maximum(
    reorder_pred,
    0
)

reorder_mae = mean_absolute_error(
    y_reorder_test,
    reorder_pred
)

reorder_rmse = np.sqrt(
    mean_squared_error(
        y_reorder_test,
        reorder_pred
    )
)

reorder_r2 = r2_score(
    y_reorder_test,
    reorder_pred
)

print("XGBoost Reorder Quantity")
print("MAE :", round(reorder_mae, 3))
print("RMSE:", round(reorder_rmse, 3))
print("R2  :", round(reorder_r2, 3))
print("R2 %:", round(reorder_r2 * 100, 2))

XGBoost Reorder Quantity
MAE : 2.824
RMSE: 7.802
R2  : 0.788
R2 %: 78.84


In [148]:
#compare actual vs predicted value
comparison = model_df.loc[
    test_mask,
    ["SKU", "Date"]
].copy()

comparison["Actual_7_Day_Demand"] = (
    y_demand_test.values
)

comparison["Predicted_7_Day_Demand"] = (
    np.round(demand_pred, 2)
)

comparison["Actual_Reorder_Qty"] = (
    y_reorder_test.values
)

comparison["Predicted_Reorder_Qty"] = (
    np.round(reorder_pred, 2)
)

comparison["Actual_Stockout_Risk"] = (
    y_risk_test.values
)

comparison["Predicted_Stockout_Risk"] = (
    risk_pred
)

comparison["Stockout_Probability"] = (
    np.round(risk_probability * 100, 2)
)

display(
    comparison.head(20)
)

,SKU,Date,Actual_7_Day_Demand,Predicted_7_Day_Demand,Actual_Reorder_Qty,Predicted_Reorder_Qty,Actual_Stockout_Risk,Predicted_Stockout_Risk,Stockout_Probability
584,SKU001,2025-08-07,85.0,90.339996,0.0,0.00,0,0,0.0
585,SKU001,2025-08-08,96.0,93.989998,0.0,0.00,0,0,0.0
586,SKU001,2025-08-09,87.0,92.860001,0.0,0.00,0,0,0.0
587,SKU001,2025-08-10,90.0,89.779999,0.0,0.00,0,0,0.0
588,SKU001,2025-08-11,84.0,92.669998,0.0,0.00,0,0,0.0
589,SKU001,2025-08-12,77.0,91.669998,0.0,0.00,0,0,0.0
590,SKU001,2025-08-13,76.0,92.760002,0.0,0.00,0,0,0.0
591,SKU001,2025-08-14,88.0,92.440002,0.0,0.00,0,0,0.0
592,SKU001,2025-08-15,87.0,93.199997,0.0,0.00,0,0,0.0
593,SKU001,2025-08-16,92.0,90.300003,0.0,0.00,0,0,0.0


In [149]:
import pickle

In [169]:
#save model
pickle.dump(reorder_model, open("recorder_quantity_model.pkl", "wb"))
pickle.dump(risk_model, open("risk_model.pkl", "wb"))
pickle.dump(demand_model, open("demand_forecast_model.pkl", "wb"))
pickle.dump(ColumnTransformer, open("column_transformer.pkl", "wb"))